In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.kernel_approximation import RBFSampler

In [ ]:
import glob

files = glob.glob("../data/eCO2mix_RTE_Annuel-Definitif_*.xls")

dfs = []
for f in files:
    df_temp = pd.read_csv(f, sep="\t", encoding="latin-1", index_col=False)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

In [ ]:
# Drop inutiles
df = df.drop(columns=["Périmètre", "Nature"])

# Datetime
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Heures"])
df["hour_sin"] = np.sin(2 * np.pi * df["datetime"].dt.hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["datetime"].dt.hour / 24)

df["day_sin"] = np.sin(2 * np.pi * df["datetime"].dt.dayofweek / 7)
df["day_cos"] = np.cos(2 * np.pi * df["datetime"].dt.dayofweek / 7)

df = df.drop(columns=["datetime"])
df = df.drop(columns=["Date", "Heures"])

# Convert object → numeric
for col in df.select_dtypes(include="object").columns:
    df[col] = pd.to_numeric(df[col].replace("ND", np.nan), errors="coerce")

# Fill NaN
df = df.fillna(df.median())


In [ ]:
print(df.columns)

cols_to_drop = [
    "Prévision J",
    "Prévision J-1",
    "Consommation",
    "Fioul",
    "Charbon",
    "Gaz",
    "Nucléaire",
    "Eolien",
    "Solaire",
    "Hydraulique",
    "Pompage",
    "Bioénergies"
]

X = df.drop(columns=cols_to_drop)
y = df["Consommation"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
models = {
    "RBF (SVR)": Pipeline([
        ("scaler", StandardScaler()),
        ("rbf", RBFSampler(gamma=0.1, n_components=500)),
        ("ridge", Ridge())
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=5))
    ]),

    "Decision Tree": DecisionTreeRegressor(random_state=42),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    # Accuracy custom (±5%)
    acc = np.mean(np.abs((y_true - y_pred) / y_true) < 0.05)

    return r2, rmse, mape, acc

In [ ]:
results = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2, rmse, mape, acc = compute_metrics(y_test, y_pred)

    results.append({
        "Model": name,
        "R2": r2,
        "RMSE": rmse,
        "MAPE": mape,
        "Accuracy (±5%)": acc
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values(by="R2", ascending=False))